# Experiment 2.18: Adaptive Newton-Schulz Iteration Count

## Counterpart identity and honest scope

This notebook is the literate counterpart to **`adaptive_ns_steps.py`** in the same experiment directory. In this first completion pass, the notebook deliberately **imports and uses the script's returned results** instead of re-implementing the training core.

### What this notebook is and is not
- **Is**: a reproducible, inspectable presentation of the current toy single-seed schedule comparison.
- **Is not**: a multi-seed statistical study or a universal validation of rank-decay / gauge-fixing claims.
- The tracked effective-rank trajectories shown here are **first-layer gradient effective-rank histories only**.
- The counted NS matmuls are an **NS-only proxy** for orthogonalization work, **not** full training compute.

### Scientific question retained from the script
In this toy setup, can lower or adaptive Newton-Schulz iteration counts preserve or improve final loss while reducing **counted Newton-Schulz-only matmul usage** relative to fixed `k=5`?

In [ ]:
from pathlib import Path
import importlib.util
import platform
import sys
import time

import matplotlib
try:
    matplotlib.use('module://matplotlib_inline.backend_inline')
except Exception:
    pass
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    from IPython.display import Markdown, display
except Exception:
    def display(obj):
        print(obj)
    def Markdown(text):
        return text

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)


def locate_experiment_dir(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    relative_script = Path('experiments/Tier_1_Core_Mechanism_Tests/ADAPTIVE_NS_steps/adaptive_ns_steps.py')

    for base in [start, *start.parents]:
        direct = base / 'adaptive_ns_steps.py'
        nested = base / relative_script
        if direct.exists():
            return direct.parent
        if nested.exists():
            return nested.parent

    raise FileNotFoundError('Could not locate ADAPTIVE_NS_steps/adaptive_ns_steps.py from the current working tree.')


EXPERIMENT_DIR = locate_experiment_dir()
SCRIPT_PATH = EXPERIMENT_DIR / 'adaptive_ns_steps.py'

spec = importlib.util.spec_from_file_location('adaptive_ns_steps', SCRIPT_PATH)
adaptive_ns_steps = importlib.util.module_from_spec(spec)
spec.loader.exec_module(adaptive_ns_steps)

print(f'Experiment directory: {EXPERIMENT_DIR}')
print(f'Script counterpart:   {SCRIPT_PATH.name}')
print('Notebook counterpart: run_experiment.ipynb')
print(f'Matplotlib backend:   {matplotlib.get_backend()}')

## Reproducibility, runtime, and configuration

The next cell executes the script-side `run_experiment()` function with the default toy configuration and logs environment/config details that matter for reproducibility. This notebook intentionally keeps the original default setup unless explicitly changed in the script.

Important calibration points before reading the outputs:
- **Single-seed only** in this pass.
- **Tracked erank and k histories are first-layer only** in the main trajectory figure.
- **Counted NS matmuls** only count the quintic Newton-Schulz update matmuls (`4 × k` per layer per step).
- No claim below should be read as a full-network compute audit or a universal theory confirmation.

In [ ]:
run_start = time.perf_counter()
results = adaptive_ns_steps.run_experiment(verbose=False)
notebook_elapsed_seconds = time.perf_counter() - run_start

run_info_df = pd.DataFrame([
    {'item': 'run_timestamp_utc', 'value': pd.Timestamp.utcnow().isoformat()},
    {'item': 'current_working_directory', 'value': str(Path.cwd())},
    {'item': 'experiment_directory', 'value': str(EXPERIMENT_DIR)},
    {'item': 'script_path', 'value': str(SCRIPT_PATH)},
    {'item': 'python_version', 'value': sys.version.split()[0]},
    {'item': 'platform', 'value': platform.platform()},
    {'item': 'numpy_version', 'value': np.__version__},
    {'item': 'pandas_version', 'value': pd.__version__},
    {'item': 'matplotlib_version', 'value': matplotlib.__version__},
    {'item': 'matplotlib_backend', 'value': matplotlib.get_backend()},
    {'item': 'script_reported_elapsed_seconds', 'value': round(results['runtime']['elapsed_seconds'], 3)},
    {'item': 'notebook_cell_elapsed_seconds', 'value': round(notebook_elapsed_seconds, 3)},
])

display(run_info_df)

architecture_config_df = pd.DataFrame([
    {
        'architecture': 'Deep Linear',
        **{k: v for k, v in results['config']['deep_linear'].items() if k not in {'display_name', 'short_name'}},
    },
    {
        'architecture': 'ReLU',
        **{k: v for k, v in results['config']['relu'].items() if k not in {'display_name', 'short_name'}},
    },
])

display(architecture_config_df)

display(pd.DataFrame([
    {'caveat_key': key, 'detail': value}
    for key, value in results['caveats'].items()
]))

## Final-loss and counted-NS-matmul summary

The table below is the main per-schedule summary for both architectures. It keeps the original toy comparison but uses more honest labels:
- **`counted_ns_matmuls`** = counted matmuls from the Newton-Schulz update only.
- **`loss_x_counted_ns_matmuls`** = a toy proxy score, useful for rough tradeoff comparisons but **not** a substitute for wall-clock or full FLOP accounting.
- The `vs_k5_*` columns use fixed `k=5` as the within-architecture reference baseline.

In [ ]:
def make_summary_frame(results_dict, architecture_key):
    payload = results_dict['architectures'][architecture_key]
    df = pd.DataFrame(payload['summary_table']).copy()
    df.insert(0, 'architecture', payload['short_name'])
    df['schedule'] = pd.Categorical(df['schedule'], categories=adaptive_ns_steps.SCHEDULE_ORDER, ordered=True)
    return df.sort_values('schedule').reset_index(drop=True)


summary_df = pd.concat([
    make_summary_frame(results, 'deep_linear'),
    make_summary_frame(results, 'relu'),
], ignore_index=True)

summary_display = summary_df[[
    'architecture',
    'schedule',
    'final_loss',
    'counted_ns_matmuls',
    'loss_x_counted_ns_matmuls',
    'vs_k5_loss_pct',
    'vs_k5_counted_ns_matmuls_pct',
    'first_layer_k_mean',
]].copy()

summary_display['final_loss'] = summary_display['final_loss'].map(lambda x: f'{x:.6e}')
summary_display['loss_x_counted_ns_matmuls'] = summary_display['loss_x_counted_ns_matmuls'].map(lambda x: f'{x:.4e}')
summary_display['vs_k5_loss_pct'] = summary_display['vs_k5_loss_pct'].map(lambda x: f'{x:+.2f}%')
summary_display['vs_k5_counted_ns_matmuls_pct'] = summary_display['vs_k5_counted_ns_matmuls_pct'].map(lambda x: f'{x:+.2f}%')
summary_display['first_layer_k_mean'] = summary_display['first_layer_k_mean'].map(lambda x: f'{x:.2f}')

display(summary_display)

## Training dynamics: loss, tracked first-layer erank, and tracked first-layer `k(t)`

This figure is the main trajectory view for the current pass.

Interpretation guide:
- **Top row**: loss trajectories.
- **Middle row**: tracked **first-layer** gradient effective-rank histories.
- **Bottom row**: tracked **first-layer** `k(t)` histories.

### Explicit caveat
The erank and `k(t)` traces below are **not** all-layer or full-network summaries. They correspond to the tracked first layer only. A small additional layerwise checkpoint summary for the adaptive-erank schedule is shown in the next section to make that limitation more honest.

In [ ]:
fig, axes = adaptive_ns_steps.plot_training_dynamics(results)
display(fig)
plt.close(fig)

## Additional honesty check: adaptive-erank layerwise `k` summary at checkpoints

The adaptive-erank schedule is chosen from each layer's gradient, not just the tracked first layer. To avoid over-reading the first-layer `k(t)` curve, the tables below summarize the adaptive-erank schedule across layers at representative checkpoints:
- `first_layer_k`: the plotted tracked-layer schedule value
- `layer_k_min`: minimum `k` used across layers at that step
- `layer_k_mean`: mean `k` across layers at that step
- `layer_k_max`: maximum `k` used across layers at that step

In [ ]:
for architecture_key in ['deep_linear', 'relu']:
    payload = results['architectures'][architecture_key]
    adaptive_erank = payload['results_by_schedule']['(e) Adaptive-erank']
    checkpoint_df = pd.DataFrame(adaptive_erank['layer_k_checkpoint_summary'])
    display(Markdown(f"### {payload['display_name']}: adaptive-erank layerwise checkpoint summary"))
    display(checkpoint_df)

## T1 / T2 / T3 / T4 checks, with caveats

The script now returns the current toy hypothesis checks in structured form. They are still **single-seed diagnostics**, not statistical tests.

- **T1**: adaptive-linear improves final loss over fixed `k=5` by more than 3%.
- **T2**: adaptive schedules improve the toy `loss × counted_ns_matmuls` proxy by more than 10%.
- **T3**: the tracked **first-layer** gradient erank under the `k=5` reference run decreases from start to end.
- **T4**: adaptive schedules use fewer **counted NS matmuls** than fixed `k=5`.

The caveat column matters: these checks intentionally avoid claiming more than is actually measured.

In [ ]:
hypothesis_df = pd.concat([
    pd.DataFrame(results['architectures']['deep_linear']['hypothesis_checks']).assign(architecture='Deep Linear'),
    pd.DataFrame(results['architectures']['relu']['hypothesis_checks']).assign(architecture='ReLU'),
], ignore_index=True)

hypothesis_display = hypothesis_df[[
    'architecture', 'test_id', 'subject', 'passed', 'metric_name', 'metric_value', 'threshold', 'description', 'caveat'
]].copy()

hypothesis_display['passed'] = hypothesis_display['passed'].map(lambda x: 'PASS' if x else 'FAIL')
hypothesis_display['metric_value'] = hypothesis_display['metric_value'].map(lambda x: f'{x:+.4f}')
hypothesis_display['threshold'] = hypothesis_display['threshold'].map(lambda x: f'{x:+.4f}')

display(hypothesis_display)

## Pareto-style proxy view

This is a compact visual summary of final loss versus **counted NS-only** matmul usage. It is useful for seeing whether a schedule is plausibly attractive in this toy accounting sense, but it should not be treated as a full efficiency result until wall-clock and broader FLOP accounting are added.

In [ ]:
fig, axes = adaptive_ns_steps.plot_pareto_summary(results)
display(fig)
plt.close(fig)

## Calibrated interpretation and conclusion

The notebook should end by clearly separating the strongest positive toy result from the weaker or mixed result, while also respecting what was *actually* measured.

In [ ]:
deep = results['architectures']['deep_linear']['results_by_schedule']
relu = results['architectures']['relu']['results_by_schedule']

deep_k5 = deep['(b) Fixed k=5']
deep_adl = deep['(d) Adaptive-linear']
deep_ader = deep['(e) Adaptive-erank']
relu_k5 = relu['(b) Fixed k=5']
relu_adl = relu['(d) Adaptive-linear']
relu_ader = relu['(e) Adaptive-erank']

deep_adl_loss_improvement = (deep_k5['final_loss'] - deep_adl['final_loss']) / deep_k5['final_loss'] * 100.0
deep_adl_matmul_saving = (deep_k5['counted_ns_matmuls'] - deep_adl['counted_ns_matmuls']) / deep_k5['counted_ns_matmuls'] * 100.0
deep_ader_loss_improvement = (deep_k5['final_loss'] - deep_ader['final_loss']) / deep_k5['final_loss'] * 100.0

relu_adl_loss_delta = (relu_adl['final_loss'] - relu_k5['final_loss']) / relu_k5['final_loss'] * 100.0
relu_adl_matmul_saving = (relu_k5['counted_ns_matmuls'] - relu_adl['counted_ns_matmuls']) / relu_k5['counted_ns_matmuls'] * 100.0
relu_ader_loss_delta = (relu_ader['final_loss'] - relu_k5['final_loss']) / relu_k5['final_loss'] * 100.0

conclusion_md = f"""
### Bottom line for this first completion pass

**Deep Linear result (strong within this toy seed):**
- The **adaptive-linear** schedule is the standout result in the current deep-linear run.
- Relative to fixed `k=5`, it lowers final loss by about **{deep_adl_loss_improvement:.1f}%** while also reducing counted NS matmuls by about **{deep_adl_matmul_saving:.1f}%**.
- The **adaptive-erank** schedule also improves final loss in this deep-linear run (about **{deep_ader_loss_improvement:.1f}%** better than `k=5`) while using fewer counted NS matmuls.
- So the deep-linear evidence in this seed is **genuinely supportive of schedule tapering** in this toy setting.

**ReLU result (mixed, not a clean win):**
- In the ReLU run, fixed `k=5` remains the best final-loss baseline among the schedules compared here.
- **Adaptive-linear** uses fewer counted NS matmuls (about **{relu_adl_matmul_saving:.1f}%** fewer) but finishes with **worse final loss** (about **{relu_adl_loss_delta:.1f}%** higher) than fixed `k=5`.
- **Adaptive-erank** is also worse than `k=5` on final loss in this seed (about **{relu_ader_loss_delta:.1f}%** higher), even though it saves counted NS matmuls.
- So the ReLU result is **a compute-saving tradeoff, not a clean optimization improvement**.

**Why the notebook now avoids overclaiming:**
- The displayed erank trajectories are **first-layer only**, and the current T3 check does **not** show a clean decrease in the `k=5` reference trace for either architecture.
- The counted NS matmuls are an **NS-only proxy**, not whole-training compute.
- These are still **single-seed toy observations** and therefore do not justify universal claims about rank decay, gauge fixing, or practical superiority across architectures.

**Calibrated conclusion:**
- The present notebook supports a **strong deep-linear toy result** for adaptive scheduling, especially for adaptive-linear tapering.
- It supports only a **mixed ReLU result**: reduced counted NS work can come with worse final loss.
- The right next step is **multi-seed robustness plus richer diagnostics**, not stronger theoretical rhetoric.
"""

display(Markdown(conclusion_md))